# Tutorial: Foundational Neural Quantum States with Disorder
## Training & Testing a ViT-based ansatz on the disordered Ising model

---

### Learning objectives

By the end of this tutorial you will be able to:

1. **Understand the physics** — define a 1D transverse-field Ising model with a uniform but random global field $h_0$.
2. **Build a Foundational NQS** — use `netket_foundational` (nkf) to train a single ViT ansatz simultaneously on many disorder realizations.
3. **Run the optimization** — set up a Natural-Gradient VMC loop with `nkf.VMC_NG`.
4. **Evaluate the results** — compare energies against exact diagonalization (ED).
5. **Diagnose quality** — plot the **V-score** and **R̂ (Rhat)** convergence diagnostics.

---

### Prerequisites

```
pip install netket netket_foundational flax optax einops scipy matplotlib pandas tqdm
```

This notebook runs comfortably on a single GPU (tested on A100). For CPU-only runs, reduce `L`, `N` and `n_iter` as suggested in the comments.

---
## Part 0 — Background

### 0.1  The transverse-field Ising model with a random global field

We study the 1D quantum Ising chain with a **uniform but randomly drawn transverse field** $h_0$:

$$
H(h_0) = -h_0 \sum_{i=1}^{L} \sigma^x_i  \;-\; J\sum_{i=1}^{L} \sigma^z_i\,\sigma^z_{i+1}
$$

with periodic boundary conditions. Each disorder **realization** is a single scalar $h_0 \sim \mathcal{U}(0, h_{\max})$, shared by all sites.  
In the code, the parameter vector passed to the network is the constant vector $\mathbf{h} = (h_0, h_0, \dots, h_0) \in \mathbb{R}^L$ — all entries are equal.  
The scalar $h_0$ is the **Hamiltonian parameter** our ansatz is conditioned on: we want to learn $|\psi_0(h_0)\rangle$ for many values of $h_0$ simultaneously.

### 0.2  Foundational NQS — the key idea

A *standard* NQS $\psi_\theta(\boldsymbol{\sigma})$ parametrizes one wave function.  
A **Foundational NQS** extends this to a *family* of wave functions:

$$
\psi_\theta(\boldsymbol{\sigma};\, \mathbf{h})
$$

The Hamiltonian parameter $h_0$ is broadcast into a constant vector and concatenated to the spin configuration before being fed to the network, so the same weights $\theta$ serve all values of $h_0$ simultaneously.  
Training is done jointly over $N$ replicas, each with its own $h_0^{(r)}$.

### 0.3  The ViT ansatz

`ViTFNQS` uses a Vision-Transformer-like encoder:

```
spins (L,)  ──┐
              ├─► patch & embed ─► Encoder (L attention blocks) ─► OutputHead ─► log ψ ∈ ℂ
params (n,) ──┘
```

Each spin-patch and the disorder parameters are concatenated and linearly embedded into tokens of dimension `d_model`. The attention mechanism (`FMHA`) then mixes information across tokens.

---
## Part 1 — Setup

In [1]:
import os
# Enable NetKet experimental sharding (multi-device parallelism)
os.environ["NETKET_EXPERIMENTAL_SHARDING"] = "1"
# Prevent JAX from pre-allocating 90 % of GPU memory
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from tqdm import tqdm
import optax
import netket as nk
import netket_foundational as nkf
from netket_foundational._src.model.vit import ViTFNQS

print("JAX devices:", jax.devices())
print("NetKet version:", nk.__version__)

/users/eleves-a/2024/rami.chagnaud/.conda/envs/env_netket/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


∣NK⟩ Tip: With many Markov Chains (e.g GPUs), n_discard_per_chain>5 is often inefficient.

JAX devices: [CudaDevice(id=0)]
NetKet version: 3.21.0


---
## Part 2 — Physical system and disorder generation

### 2.1  Hilbert space and parameter space

In [ ]:
# ── Physical parameters ──────────────────────────────────────────────────────
L    = 16      # System size  (reduce to 8 for CPU-only runs)
h0   = 1.0     # Uniform upper bound on the random fields
J    = 1.0     # Ising coupling (here J = 1/e ≈ 0.368 in the original paper;
               # we use J = 1.0 for simplicity)
N    = 20      # Number of disorder realizations trained simultaneously
               # (reduce to 8 on CPU)

seed = 42       
rng  = np.random.default_rng(seed)
k    = jax.random.key(seed)

# Hilbert space: L spin-1/2 sites
hi = nk.hilbert.Spin(0.5, L)
print(f"Hilbert space dimension: 2^{L} = {hi.n_states}")

# ParameterSpace: tells nkf the shape and range of the Hamiltonian parameters.
# Even though the physical parameter is a single scalar h0, we pass it as a
# constant vector of size L (one entry per site, all equal to h0).
# This is the format expected by ViTFNQS with disorder=True.
ps = nkf.ParameterSpace(N=hi.size, min=0, max=h0)
print(f"Parameter space dimension: {ps.size}  (= L = {hi.size}, all entries = h0)")

### 2.2  Generating disorder realizations

Each disorder realization is a single scalar $h_0^{(r)} \sim \mathcal{U}(0, h_{\max})$.  
It is then broadcast into a **constant vector** $(h_0^{(r)}, \dots, h_0^{(r)}) \in \mathbb{R}^L$ — all sites share the same field value.  
We generate `N` such vectors to form the training set.

In [ ]:
def generate_disorder(N_real, system_size, h_max, rng=None):
    """Return an array of shape (N_real, system_size).
    Each realization draws a single scalar h0 ~ U(0, h_max),
    then broadcasts it to all L sites: params[r] = [h0, h0, ..., h0].
    """
    if rng is None:
        rng = np.random.default_rng()
    h0_values = rng.uniform(0.0, h_max, size=N_real)   # one scalar per realization
    return np.tile(h0_values[:, None], (1, system_size))  # broadcast to (N_real, L)

params_train = generate_disorder(N, hi.size, h0, rng=rng)
print(f"Training disorder realizations shape: {params_train.shape}")
print("First realization — all entries equal to h0 =", params_train[0, 0].round(3))
assert np.allclose(params_train[0], params_train[0, 0]), "All sites must share the same h0"

### 2.3  Defining the Hamiltonian

We use `nkf.operator.ParametrizedOperator`, which wraps a constructor function `create_operator` that takes a parameter vector and returns a NetKet operator.

In [ ]:
def create_operator(params):
    """Build H(h0) for a given realization.
    params = (h0, h0, ..., h0): a constant vector, all entries equal to h0.
    """
    assert params.shape == (hi.size,), f"Expected shape ({hi.size},), got {params.shape}"
    h0_val = params[0]   # all entries are equal, we just read the first one

    # Transverse-field term:  -h0 ∑_i σ^x_i
    ha_X = h0_val * sum(
        nkf.operator.sigmax(hi, i)
        for i in range(hi.size)
    )

    # Ising interaction:  -J ∑_i σ^z_i σ^z_{i+1}   (periodic BC)
    ha_ZZ = sum(
        nkf.operator.sigmaz(hi, i) @ nkf.operator.sigmaz(hi, (i + 1) % hi.size)
        for i in range(hi.size)
    )

    return -ha_X - J * ha_ZZ

# ParametrizedOperator automatically dispatches create_operator to all replicas
ha_p  = nkf.operator.ParametrizedOperator(hi, ps, create_operator)

# Magnetization observable  Mz = (1/L) ∑_i σ^z_i
Mz    = sum(nkf.operator.sigmaz(hi, i) for i in range(hi.size)) * (1.0 / hi.size)
mz_p  = nkf.operator.ParametrizedOperator(hi, ps, lambda _: Mz)

# Quick sanity check: build the operator for the first realization
H0 = create_operator(params_train[0])
print("Operator built for first realization:", H0)

---
## Part 3 — Building the Foundational NQS

### 3.1  The ViTFNQS model

Key hyper-parameters:

| Parameter | Role |
|-----------|------|
| `b` | patch size — spins are grouped in windows of size `b` before embedding |
| `L_eff` | effective sequence length = `L // b` |
| `d_model` | token embedding dimension |
| `heads` | number of attention heads in FMHA |
| `num_layers` | depth of the Transformer encoder |
| `n_coups` | size of the disorder parameter vector appended to each token |
| `disorder=True` | uses a dedicated embedding for the disorder parameters |
| `transl_invariant=False` | disorder breaks translation symmetry, so we disable it |
| `complex=True` | the ansatz outputs a complex amplitude $\log\psi \in \mathbb{C}$ |

In [ ]:
b     = 4         # patch size   (must divide L)
L_eff = L // b    # effective sequence length

ma = ViTFNQS(
    num_layers       = 2,       # number of Transformer encoder blocks
    d_model          = 16,      # embedding dimension
    heads            = 4,       # attention heads
    b                = b,       # patch size
    L_eff            = L_eff,   # effective sequence length
    n_coups          = ps.size, # appended disorder vector length
    complex          = True,    # complex-valued ansatz
    disorder         = True,    # use disorder-specific embedding
    transl_invariant = False,   # no translation symmetry (disorder breaks it)
    two_dimensional  = False,   # 1D chain
)

print("Model architecture summary:")
print(f"  Patch size b      = {b}")
print(f"  Effective length  = {L_eff}  tokens")
print(f"  Token dimension   = {ma.d_model}")
print(f"  Disorder concat   = {ma.n_coups} params per token")

### 3.2  Sampler and Foundational Quantum State

`FoundationalQuantumState` (nkf) is the core object:  
it wraps a single set of neural-network weights **plus** an array of $N$ disorder realizations and runs MCMC sampling and gradient estimation simultaneously for all replicas.

In [ ]:
# ── Sampler ──────────────────────────────────────────────────────────────────
# MetropolisLocal proposes single-spin flips — a good default for spin systems.
n_chains_per_replica = 16
n_chains = N * n_chains_per_replica          # total chains across all replicas
n_samples = N * n_chains_per_replica * 4     # 4 samples per chain per step

sa = nk.sampler.MetropolisLocal(hi, n_chains=n_chains)

# ── Variational state ────────────────────────────────────────────────────────
vs = nkf.FoundationalQuantumState(
    sa,
    ma,
    ps,
    n_replicas = N,
    n_samples  = n_samples,
    seed       = seed,
)

# Assign disorder realizations
vs.parameter_array = params_train
print(f"FoundationalQuantumState ready: {N} replicas, {n_samples} total samples/step")

---
## Part 4 — Optimization

We use **Natural Gradient VMC** (`nkf.VMC_NG`) with a linear learning-rate schedule and a diagonal-shift regularization of the quantum geometric tensor.

> 💡 `VMC_NG` computes the quantum natural gradient efficiently using the stochastic reconfiguration method. The `diag_shift` parameter adds $\epsilon\,\mathbf{I}$ to the QFI matrix to avoid singularities.

In [ ]:
n_iter     = 300     # number of optimization steps (reduce to 100 on CPU)
lr_init    = 0.03
lr_end     = 0.005
diag_shift = 1e-4

# Linear learning-rate decay
learning_rate = optax.linear_schedule(
    init_value      = lr_init,
    end_value       = lr_end,
    transition_steps= n_iter,
)
optimizer = optax.sgd(learning_rate)

gs = nkf.VMC_NG(
    ha_p,
    optimizer,
    variational_state = vs,
    diag_shift        = diag_shift,
)

# ── Logging ──────────────────────────────────────────────────────────────────
os.makedirs("output", exist_ok=True)
log = nk.logging.JsonLog("output/log", save_params=True)

print("Optimizer set up. Starting training...")

In [ ]:
# ── Run ──────────────────────────────────────────────────────────────────────
# obs: additional observables measured at every logged step (every step_size iterations)
gs.run(
    n_iter,
    out  = log,
    obs  = {"ham": ha_p, "mz": mz_p},
    step_size = 10,   # log every 10 steps
)
print("Training complete.")

---
## Part 5 — Convergence diagnostics

### 5.1  Energy convergence curves

The `log.data` dictionary stores, for each observable, one time-series per replica.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

for r in range(N):
    e_log = log.data["Energy"]      # shape: (n_logged_steps, N)
    # log.data stores each replica separately; index by replica r
    replica_log = log.data["ham"][r]
    ax.plot(replica_log.iters, np.real(replica_log.Mean), alpha=0.4, linewidth=0.8)

ax.set_xlabel("Iteration")
ax.set_ylabel("Energy")
ax.set_title("Training energy — all disorder realizations")
ax.set_xscale("log")
fig.tight_layout()
plt.savefig("output/convergence.pdf")
plt.show()

With a variational state, you can compute expectation values of operators.
Notice that it also provides an error estimate and the variance of this estimator.
If you are close to an eigenstate of the operators, the variance should be 0 or close to 0.

The $\hat{R}$ value is a Monte-Carlo convergence estimator. It will be $\hat{R}\approx 1$ if the Markov Chain is converged, while it will be far from $1$ if your sampling has not converged.
As a rule of thumb, look out for $|\hat{R}| > 1.1$, and check if your sampling scheme or sampler is consistent with your system specification.

You can also investigate the correlation time of your estimator, $\tau$. If $\tau\gg1$ then your samples are very correlated and you most likely have some issues with your sampling scheme.

In [ ]:
# Compute R̂ for each replica on the final variational state
rhat_values = []

for r in tqdm(range(N), desc="Computing R̂"):
    pars = vs.parameter_array[r]
    _vs  = vs.get_state(pars)          # extract single-replica variational state

    # Create a small MCState for this replica
    vs_mc = nk.vqs.MCState(
        sampler   = nk.sampler.MetropolisLocal(hi, n_chains=32),
        model     = _vs.model,
        variables = _vs.variables,
        n_samples = 512,
        chunk_size= 64,
    )

    stats = vs_mc.expect(create_operator(pars))
    rhat_values.append(float(np.real(stats.R_hat)))

rhat_values = np.array(rhat_values)

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(range(N), rhat_values, color="steelblue", alpha=0.8)
ax.axhline(1.1, color="red", linestyle="--", label="R̂ = 1.1 threshold")
ax.set_xlabel("Replica index")
ax.set_ylabel("R̂")
ax.set_title("Gelman–Rubin R̂ per disorder realization")
ax.legend()
fig.tight_layout()
plt.savefig("output/rhat.pdf")
plt.show()

print(f"Mean R̂ = {rhat_values.mean():.4f}  |  Max R̂ = {rhat_values.max():.4f}")
n_bad = (rhat_values > 1.1).sum()
print(f"Replicas with R̂ > 1.1 : {n_bad}/{N}")

---
## Part 6 — Comparison with exact diagonalization

For $L \le 20$ we can compute the exact ground-state energy with NetKet's Lanczos solver and compare it to our VMC estimate.

In [ ]:
exact_energies = []
vmc_energies   = []
rel_errors     = []

for r, pars in tqdm(enumerate(vs.parameter_array), total=N, desc="ED comparison"):
    _ha  = create_operator(pars)
    E_ed = nk.exact.lanczos_ed(_ha, k=1, compute_eigenvectors=False).item()

    # VMC energy: use the full-sum state for an unbiased estimate (works for L≤20)
    _vs = vs.get_state(pars)
    vs_fs = nk.vqs.FullSumState(
        hilbert    = hi,
        model      = _vs.model,
        variables  = _vs.variables,
        chunk_size = 64,
    )
    E_vmc = float(np.real(vs_fs.expect(_ha).Mean))

    exact_energies.append(E_ed)
    vmc_energies.append(E_vmc)
    rel_errors.append(abs((E_vmc - E_ed) / E_ed))

exact_energies = np.array(exact_energies)
vmc_energies   = np.array(vmc_energies)
rel_errors     = np.array(rel_errors)

print(f"Mean relative energy error : {rel_errors.mean()*100:.3f} %")
print(f"Max  relative energy error : {rel_errors.max()*100:.3f} %")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── Panel 1 : scatter VMC vs ED ───────────────────────────────────────────────
ax = axes[0]
ax.scatter(exact_energies, vmc_energies, alpha=0.7, s=30, label="VMC vs ED")
lims = [min(exact_energies.min(), vmc_energies.min()),
        max(exact_energies.max(), vmc_energies.max())]
ax.plot(lims, lims, "k--", linewidth=1, label="identity")
ax.set_xlabel("Exact energy $E_0$")
ax.set_ylabel("VMC energy")
ax.set_title("VMC vs exact diagonalization")
ax.legend()

# ── Panel 2 : relative error histogram ───────────────────────────────────────
ax = axes[1]
ax.hist(rel_errors * 100, bins=15, color="steelblue", edgecolor="white")
ax.set_xlabel("Relative error (%)")
ax.set_ylabel("Count")
ax.set_title("Distribution of relative energy errors")

fig.tight_layout()
plt.savefig("output/ed_comparison.pdf")
plt.show()

---
## Part 7 — V-score diagnostic

### What is the V-score?

The **V-score** (Variational score) is a dimensionless, size-intensive measure of ansatz quality:

$$
V = \frac{\text{Var}[H]}{E_0^2} = \frac{\langle H^2\rangle - \langle H\rangle^2}{\langle H\rangle^2}
$$

- $V = 0$ → exact ground state ✅  
- Larger $V$ → larger variational energy uncertainty ⚠️

It is particularly useful for **comparing different ansätze** across system sizes, since it normalizes by the energy scale.

> 💡 A V-score below $10^{-3}$ is generally considered excellent for ground-state problems.

In [ ]:
v_scores = []

for r, pars in tqdm(enumerate(vs.parameter_array), total=N, desc="V-score"):
    _vs = vs.get_state(pars)

    vs_mc = nk.vqs.MCState(
        sampler   = nk.sampler.MetropolisLocal(hi, n_chains=32),
        model     = _vs.model,
        variables = _vs.variables,
        n_samples = 2048,
        chunk_size= 64,
    )

    stats = vs_mc.expect(create_operator(pars))
    E     = float(np.real(stats.Mean))
    Var   = float(np.real(stats.Variance))
    vscore = Var / (E**2 + 1e-12)      # small epsilon to avoid division by zero
    v_scores.append(vscore)

v_scores = np.array(v_scores)
print(f"Mean V-score : {v_scores.mean():.2e}")
print(f"Max  V-score : {v_scores.max():.2e}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(N), v_scores, color="darkorange", alpha=0.8)
ax.axhline(1e-3, color="red", linestyle="--", label="V-score = 10⁻³")
ax.set_yscale("log")
ax.set_xlabel("Replica index")
ax.set_ylabel("V-score")
ax.set_title("V-score per disorder realization")
ax.legend()
fig.tight_layout()
plt.savefig("output/vscore.pdf")
plt.show()

---
## Part 8 — Testing on unseen disorder realizations

A key advantage of the Foundational NQS is **zero-shot generalization**:  
we can evaluate the trained model on disorder realizations it has never seen during training,  
simply by assigning new parameter vectors to `vs.parameter_array`.

In [ ]:
N_test = 30
params_test = generate_disorder(N_test, hi.size, h0, rng=rng)  # new realizations

test_results = {"E_ed": [], "E_vmc": [], "rel_err": [], "v_score": []}

for r, pars in tqdm(enumerate(params_test), total=N_test, desc="Test"):
    _ha  = create_operator(pars)
    E_ed = nk.exact.lanczos_ed(_ha, k=1, compute_eigenvectors=False).item()

    _vs = vs.get_state(pars)   # ← same model weights, new disorder
    vs_fs = nk.vqs.FullSumState(
        hilbert    = hi,
        model      = _vs.model,
        variables  = _vs.variables,
        chunk_size = 64,
    )
    stats  = vs_fs.expect(_ha)
    E_vmc  = float(np.real(stats.Mean))
    Var    = float(np.real(stats.Variance))

    test_results["E_ed"].append(E_ed)
    test_results["E_vmc"].append(E_vmc)
    test_results["rel_err"].append(abs((E_vmc - E_ed) / E_ed))
    test_results["v_score"].append(Var / (E_vmc**2 + 1e-12))

for k_name in test_results:
    test_results[k_name] = np.array(test_results[k_name])

print(f"Test mean relative error : {test_results['rel_err'].mean()*100:.3f} %")
print(f"Test mean V-score        : {test_results['v_score'].mean():.2e}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.scatter(test_results["E_ed"], test_results["E_vmc"], alpha=0.7, color="teal", s=40)
lims = [test_results["E_ed"].min() * 1.02, test_results["E_ed"].max() * 0.98]
ax.plot(lims, lims, "k--", linewidth=1)
ax.set_xlabel("Exact $E_0$")
ax.set_ylabel("VMC energy (test)")
ax.set_title("Generalization: VMC vs ED on unseen realizations")

ax = axes[1]
ax.bar(range(N_test), test_results["v_score"], color="teal", alpha=0.8)
ax.axhline(1e-3, color="red", linestyle="--")
ax.set_yscale("log")
ax.set_xlabel("Test replica")
ax.set_ylabel("V-score")
ax.set_title("V-score on test set")

fig.tight_layout()
plt.savefig("output/test_results.pdf")
plt.show()

METTRE LE GRAPHE D'EXEMPLE POUR QU'ILS PUISSENT COMPARER